# BERT Model Training & Evaluation

This notebook trains and evaluates the BERT model for fake news detection.

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import sys
sys.path.insert(0, '../backend')

from models.bert_model import BERTModel
from src.data_loader import FakeNewsDataLoader

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## 1. Load and Prepare Data

In [ ]:
# Load data
loader = FakeNewsDataLoader()
data = loader.load_data()

# Split data
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data['label']
)

# Create labels mapping
label2id = {'FAKE': 1, 'REAL': 0}
id2label = {v: k for k, v in label2id.items()}

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Label mapping: {label2id}")

## 2. Initialize BERT Model

In [ ]:
# Initialize BERT model
model = BERTModel(
    model_name='bert-base-uncased',
    num_labels=2,
    max_length=512
)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model moved to device: {device}")
print(f"Model loaded successfully")

## 3. Prepare Dataset

In [ ]:
from datasets import Dataset

# Create dataset format for Hugging Face
train_dataset = Dataset.from_dict({
    'text': train_data['text'].tolist(),
    'label': train_data['label'].map(label2id).tolist()
})

test_dataset = Dataset.from_dict({
    'text': test_data['text'].tolist(),
    'label': test_data['label'].map(label2id).tolist()
})

# Tokenize datasets
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f"Datasets prepared!")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

## 4. Training Configuration

In [ ]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'f1': f1_score(labels, predictions)
    }

# Training arguments
training_args = TrainingArguments(
    output_dir='../models/bert_checkpoint',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1'
)

print("Training configuration:")
print(f"  Epochs: 5")
print(f"  Batch size: 8")
print(f"  Warmup steps: 500")

## 5. Train Model

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train
train_result = trainer.train()
print("Training completed!")

## 6. Evaluate on Test Set

In [ ]:
# Evaluate
eval_results = trainer.evaluate(eval_dataset=test_dataset)

print("\nBERT Model - Test Set Performance:")
print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_recall']:.4f}")
print(f"  F1-Score:  {eval_results['eval_f1']:.4f}")

## 7. Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Confusion matrix
cm = confusion_matrix(labels, preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['REAL', 'FAKE'],
            yticklabels=['REAL', 'FAKE'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('BERT Model - Confusion Matrix')
plt.tight_layout()
plt.show()

print(f"\nTrue Negatives:  {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives:  {cm[1, 1]}")

## 8. Save Model

In [ ]:
# Save model
model.save_pretrained('../models/bert_model')
tokenizer.save_pretrained('../models/bert_model')
print("BERT model and tokenizer saved!")